# Earthquake Signal Classification

This notebook walks through the reproducible portfolio pipeline for classifying waveform segments into `noise`, `microseisms`, and `earthquakes`.

It uses the modular code in `src/` rather than re-implementing the workflow inline.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve() / "labs" / "seismic_signal_classification"
if not PROJECT_ROOT.exists():
    PROJECT_ROOT = Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.pipeline import run_seismic_classification_pipeline
from src.visualization import (
    plot_average_spectra,
    plot_confusion_matrix,
    plot_example_waveforms,
    plot_feature_importance,
    plot_scalogram_example,
)

artifacts = run_seismic_classification_pipeline(PROJECT_ROOT)
artifacts.model_artifacts.summary_table.round(3)

In [ ]:
import numpy as np
import pandas as pd

archive = np.load(artifacts.dataset_paths["npz"])
metadata = pd.read_csv(artifacts.dataset_paths["metadata"])
waveforms = archive["signals"]

plot_example_waveforms(waveforms, metadata, sample_rate_hz=20.0)
plot_average_spectra(waveforms, metadata, sample_rate_hz=20.0)
microseism_index = int(metadata.loc[metadata["label"] == "microseisms", "sample_index"].iloc[0])
plot_scalogram_example(waveforms[microseism_index], sample_rate_hz=20.0)

In [ ]:
best_model_name = artifacts.model_artifacts.best_model_name
best_test_eval = artifacts.model_artifacts.evaluations[f"{best_model_name}__test"]

plot_confusion_matrix(best_test_eval.confusion, f"Confusion matrix - {best_model_name}")
plot_feature_importance(artifacts.model_artifacts.feature_importances, best_model_name)

best_test_eval.per_class.round(3)